# Petta AI Care Assistant — Free GPU Inference Server (Colab + Cloudflare Tunnel)

Serves your fine-tuned **`MinamiYuki1/petta-qwen2.5-3b-lora`** adapter on a free Colab T4/L4 GPU and exposes a public HTTPS URL (`https://xxxx.trycloudflare.com`) for your Render app (`PETTA_AI_URL`).

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes fastapi uvicorn nest_asyncio
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import os, re, subprocess, threading, time
import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "MinamiYuki1/petta-qwen2.5-3b-lora"
SHARED_SECRET = os.environ.get("PETTA_AI_SECRET", "")  # optional

SYSTEM_PROMPT = (
    "You are Petta AI Care Assistant, a helpful, safety-guarded pet care guide for dog and cat owners "
    "in the Philippines. You understand English, Taglish, Tagalog, and common Philippine regional phrases. "
    "Provide clear, non-diagnostic home-care, nutrition, and hygiene guidance. Never diagnose diseases "
    "or prescribe controlled medications. Always remind owners to consult a licensed vet for persistent symptoms."
)

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, use_fast=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
print("Loaded", ADAPTER_REPO)

app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    pet_context: str = ""
    history: list[dict] = []

@app.get("/health")
def health():
    return {"status": "ok", "model": ADAPTER_REPO}

@app.post("/chat")
def chat(req: ChatRequest, x_petta_secret: str | None = Header(default=None)):
    if SHARED_SECRET and x_petta_secret != SHARED_SECRET:
        raise HTTPException(status_code=401, detail="Invalid secret")

    sys_content = SYSTEM_PROMPT
    if req.pet_context:
        sys_content += f"\n\n{req.pet_context}"

    messages = [{"role": "system", "content": sys_content}]
    for h in req.history[-6:]:
        if h.get("role") in ("user", "assistant") and h.get("content"):
            messages.append({"role": h["role"], "content": str(h["content"])})
    messages.append({"role": "user", "content": req.message})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=320,
            temperature=0.35,
            top_p=0.9,
            do_sample=True,
        )
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return {"reply": reply}

threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True,
).start()
time.sleep(2)

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in tunnel.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n======================================================")
        print("PETTA_AI_URL =", m.group(0))
        print("Paste this URL into Render -> Environment -> PETTA_AI_URL")
        print("======================================================\n")